REMOVING OUTLIERS USING ZSCORE
APPLY DECISION TREE ON BOTH CLEANED DATASET AND UNCLEANED DATASET
APPLY RANDOM FOREST CLASSIFIER ON CLEANED DATASET
EVALUATE PRECISION,F1 SCORE , RECALL SCORE ON THE THREE MODELS
DATASET USED: DIABETES

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score , recall_score , f1_score
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import zscore
from google.colab import drive
drive.mount('/content/drive')

df=pd.read_csv('/content/drive/MyDrive/SETA_diabetes - SETA_diabetes.csv')
df


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [ ]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [ ]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [ ]:
df.isnull().count()

,0
Pregnancies,768
Glucose,768
BloodPressure,768
SkinThickness,768
Insulin,768
BMI,768
DiabetesPedigreeFunction,768
Age,768
Outcome,768


In [ ]:
#using original dataset
X = df.drop('Outcome', axis=1)
y = df['Outcome']
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(614, 8)
(154, 8)
(614,)
(154,)


In [ ]:
dt=DecisionTreeClassifier()
dt.fit(x_train,y_train)
y_pred=dt.predict(x_test)
print("precision_original_dt",precision_score(y_test,y_pred))
print("recall_original_dt",recall_score(y_test,y_pred))
print("f1_original_dt",f1_score(y_test,y_pred))

precision_original_dt 0.6486486486486487
recall_original_dt 0.4444444444444444
f1_original_dt 0.5274725274725275


In [ ]:
df1=df.copy()
# Calculate Z-scores
z_scores = np.abs(zscore(df1))
outliers = (z_scores > 3).any(axis=1)

# Remove Outliers
df1 = df1[~outliers]

print(f"Cleaned training set size: {df1.shape[0]}")

Cleaned training set size: 688


In [ ]:
X1 = df1.drop('Outcome', axis=1)
y1 = df1['Outcome']
x_train1,x_test1,y_train1,y_test1=train_test_split(X1,y1,test_size=0.2,random_state=42,stratify=y1)


In [ ]:
dt=DecisionTreeClassifier()
dt.fit(x_train1,y_train1)
y_pred_clean=dt.predict(x_test1)
print("precision_cleaned_dt",precision_score(y_test1,y_pred_clean))
print("recall_cleaned_dt",recall_score(y_test1,y_pred_clean))
print("f1_cleaned_dt",f1_score(y_test1,y_pred_clean))

precision_cleaned_dt 0.6
recall_cleaned_dt 0.6521739130434783
f1_cleaned_dt 0.625


In [ ]:
#random forest on cleaned data
rf = RandomForestClassifier(random_state=42)
rf.fit(x_train1, y_train1)
rf_pred=rf.predict(x_test1)
print("precision_original_rf",precision_score(y_test1,rf_pred))
print("recall_original_rf",recall_score(y_test1,rf_pred))
print("f1_score_rf",f1_score(y_test1,rf_pred))

precision_original_rf 0.6842105263157895
recall_original_rf 0.5652173913043478
f1_score_rf 0.6190476190476191


In [ ]:
evalu=pd.DataFrame({'Precision':[precision_score(y_test1,rf_pred),precision_score(y_test1,y_pred_clean),precision_score(y_test,y_pred)],
                    'Recall':[recall_score(y_test1,rf_pred),recall_score(y_test1,y_pred_clean),recall_score(y_test,y_pred)],
                    'F1_score':[f1_score(y_test1,rf_pred),f1_score(y_test1,y_pred_clean),f1_score(y_test,y_pred)]},index=['Random_forest_original','Decision_Tree_cleaned','Decision_tree_original'])
evalu

,Precision,Recall,F1_score
Random_forest_original,0.684211,0.565217,0.619048
Decision_Tree_cleaned,0.600000,0.652174,0.625000
Decision_tree_original,0.648649,0.444444,0.527473
